## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Bajar datasets si hace falta

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo Regresion Lineal

## 1.1 Init Experimento

In [ ]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q statsmodels

In [ ]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [ ]:
import os as os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [ ]:
# defino los parametros
PARAM = {'experimento':'LR01',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':102191,
  'empiojar_ruido':0.0
}

In [ ]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Preprocesamiento

In [ ]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
display( tb_ventas )

### 1.2.1 Empiojado  lognormal

In [ ]:

if PARAM['empiojar_ruido']>0.0:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=PARAM['empiojar_ruido'], size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


### 1.2.2  Dataset aplanado con lags

In [ ]:
lags = [-2, *range(0,12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
      .with_columns(
          [
              pl.col("tn")
                .shift(lag)
                .over("product_id")
                .alias(f"tn_{lag}")
              for lag in lags
          ]
      )
)

tb_lags = tb_lags.rename({"tn_-2": "clase"})

In [ ]:
display(tb_lags)

### 1.2.4  Definicion de Training

In [ ]:
# Esta es la salsa mágica del notebook

productos_magicos = [ 20002, 20003, 20005, 20006, 20009, 20010, 20011, 20013, 20015,
  20017, 20018, 20019, 20021, 20022, 20026, 20028, 20035, 20038, 20042, 20043, 20044,
  20045, 20046, 20049, 20051, 20052, 20055, 20081, 20324, 20417, 20771, 20947, 20949, 21003
]

productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]

In [ ]:
# Entreno con los datos de 2018 para los  productos_magicos

dtrain = tb_lags.filter( (pl.col("periodo") == 201812) & (pl.col("product_id").is_in(productos_magicos)) )

display( dtrain )

## 1.3  Modelo de Regresion Lineal

In [ ]:

# campos a utilizar
campos_buenos = dtrain.select(cs.starts_with("tn_"))


# tristemente paso por pandas
X_train = dtrain.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_train = sm.add_constant(X_train)

# tristemente paso por pandas
y_train = dtrain['clase'].to_pandas()


modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

## 1.4 Aplicacion a los datos del futuro

### 1.4.1  Definicion datos del futuro
Solo puedo aplicar al modelo a los datos que tengan completos todos los meses
<br> de los 780 productos a predecir
* 656 tienen todos los meses del 2019 completos
* 124 no (los voy a predecir con el simple promedio)

In [ ]:
dfuture = tb_lags.filter( (pl.col("periodo") == 201912) & (pl.col("tn_11").is_not_null() ))

display( dfuture )

### 1.4.2  Aplicacion del modelo de regresion a los datos del futuro

Se aplica el modelo a los 656 datos del futuro

In [ ]:

# campos a utilizar
campos_buenos = dfuture.select(cs.starts_with("tn_"))

# tristemente paso por pandas
X_future = dfuture.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_future = sm.add_constant(X_future)

# la prediccion
prediccion = modelo.predict(X_future)


tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)

display( tb_regresion)

### 1.4.3  Join de los modelos de regresion y promedio

In [ ]:
# tabla conh los promedios
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter( pl.col("periodo").is_between(primer_periodo,ultimo_periodo)).group_by("product_id").agg(
 pl.col("tn").mean().alias("tn"))

tb_meses12 = tb_meses12.select(["product_id", "tn"])
display( tb_meses12 )

In [ ]:
# Update table1 using table2
tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left", suffix="_update")
    .with_columns(
        pl.coalesce([pl.col("tn_pred"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_pred")
)

display( tb_final )

## 1.5 Submit a Kaggle

In [ ]:
# Submit a Kaggle
if PARAM['empiojar_ruido']<=0.0:
  archivo= "linreg.csv"
  mensaje= "Regresion Lineal"
else:
  archivo= "linreg_empiojado.csv"
  mensaje= "Linear Regression logEMPIOJADO al " + str(PARAM['empiojar_ruido'])

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )

[![Ver video](https://www.youtube.com/embed/IY9G79BDRS4)](https://www.youtube.com/embed/IY9G79BDRS4)